In [15]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

In [16]:
x, y = fetch_california_housing(return_X_y=True, as_frame=True)

x.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [17]:
x_train, x_testval, y_train, y_testval = train_test_split(x, y, test_size=0.2, random_state=42)
x_test, x_val, y_test, y_val = train_test_split(x_testval, y_testval, test_size=0.5, random_state=42)

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [19]:
lat_lon_cols = ['Latitude', 'Longitude']
processed_cols = [c for c in x_train.columns if c not in lat_lon_cols]

Q1 = x_train[processed_cols].quantile(0.25)
Q3 = x_train[processed_cols].quantile(0.75)
IQR = Q3 - Q1

lower_bound = torch.tensor((Q1 - 1.5 * IQR).values, dtype=torch.float32)
upper_bound = torch.tensor((Q3 + 1.5 * IQR).values, dtype=torch.float32)
mean = torch.tensor(x_train[processed_cols].mean().values, dtype=torch.float32)
std = torch.tensor(x_train[processed_cols].std().values, dtype=torch.float32)

In [20]:
class ScalingLayer(nn.Module):
    def __init__(self, mean: torch.Tensor, std: torch.Tensor):
        super().__init__()
        self.register_buffer('mean', mean)
        self.register_buffer('std', std)

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        return (X - self.mean) / (self.std + 1e-8)

In [21]:
class PreprocessingLayer(nn.Module):
    processed_cols_idx = [0, 1, 2, 3, 4, 5]
    latlon_idx = [6, 7]

    def __init__(self, mean, std, lower, upper):
        super().__init__()
        self.scaler = ScalingLayer(mean, std)
        self.register_buffer('lower', lower)
        self.register_buffer('upper', upper)

    def forward(self, x):                                      
        x_main = x[:, self.processed_cols_idx]
        x_latlon = x[:, self.latlon_idx]
        x_main = torch.clamp(x_main, self.lower, self.upper) 
        x_main = self.scaler(x_main)                         
        return torch.cat([x_main, x_latlon], dim=1)

In [22]:
class HousingDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).to(device)
        self.y = torch.tensor(y, dtype=torch.float32).to(device)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [23]:
train_data = HousingDataset(x_train.values, y_train.values)
val_data   = HousingDataset(x_val.values,   y_val.values)
test_data  = HousingDataset(x_test.values,  y_test.values)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=64)
test_loader  = DataLoader(test_data,  batch_size=64)